# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedumer1941/Flyrank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/ahmedumer1941/Flyrank-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(f"Loaded {len(df)} rows x {len(df.columns)} columns")

df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"Label distribution:\n{df['is_declining'].value_counts()}")
print(f"  -> {df['is_declining'].mean()*100:.1f}% of pages are declining\n")

numeric_features = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d',
    'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

X_num = df[numeric_features].copy()

for col in ['search_volume', 'competition', 'cpc', 'word_count', 'char_count']:
    missing = X_num[col].isna().sum()
    if missing > 0:
        # Fix: Group using df['content_type'] since it's not in X_num
        X_num[col] = X_num[col].fillna(df.groupby('content_type')[col].transform('median'))
        # Secondary fill with global median for any content_types that are all NaN
        X_num[col] = X_num[col].fillna(X_num[col].median())
        print(f"Imputed {missing} missing values in '{col}' using per-content-type median")

zero_pos = (X_num['avg_position'] == 0).sum()
X_num['avg_position'] = X_num['avg_position'].replace(0, X_num['avg_position'].median())
print(f"Replaced {zero_pos} zero-avg_position rows with median ({X_num['avg_position'].median():.1f})")

# Engineered features
X_num['ctr_gap'] = (X_num['ctr'].max() - X_num['ctr']) / (X_num['ctr'].max() - X_num['ctr'].min() + 1e-6)
X_num['imp_to_session_ratio'] = X_num['sessions_90d'] / (X_num['impressions_90d'] + 1e-6)
X_num['staleness_weight'] = X_num['days_since_last_update'] / (X_num['content_age_days'] + 1e-6)
X_num['engagement_per_session'] = X_num['engaged_sessions_90d'] / (X_num['sessions_90d'] + 1e-6)
X_num['ai_ratio'] = X_num['ai_sessions_90d'] / (X_num['sessions_90d'] + 1e-6)

# Categorical features
categorical_features = ['content_type', 'main_intent', 'competition_level',
                         'age_tier', 'freshness_tier', 'impression_tier',
                         'word_count_tier', 'position_tier']
X_cat = pd.get_dummies(df[categorical_features], drop_first=True)

# Combine
feature_vector = pd.concat([df[['content_id', 'client_id']], X_num, X_cat, df[['is_declining']]], axis=1)
print(f"\nFinal feature vector: {feature_vector.shape[0]} rows x {feature_vector.shape[1]} columns")
print(f"Feature columns (excluding identifiers & label): {feature_vector.shape[1] - 3}")
feature_vector.head()

Loaded 30000 rows x 44 columns
Label distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64
  -> 54.2% of pages are declining

Imputed 2468 missing values in 'search_volume' using per-content-type median
Imputed 2468 missing values in 'competition' using per-content-type median
Imputed 2468 missing values in 'cpc' using per-content-type median
Imputed 7699 missing values in 'word_count' using per-content-type median
Imputed 7699 missing values in 'char_count' using per-content-type median
Replaced 1205 zero-avg_position rows with median (10.8)

Final feature vector: 30000 rows x 53 columns
Feature columns (excluding identifiers & label): 50


,content_id,client_id,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,impressions_90d,...,impression_tier_low,impression_tier_moderate,word_count_tier_2000-3500,word_count_tier_3500+,word_count_tier_<1000,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,0.67,2.05,3221.0,20457.0,187,20,3803,...,False,False,True,False,False,False,False,True,False,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,0.05,2481.0,15562.0,445,25,15320,...,False,False,True,False,False,False,True,False,False,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,0.00,3515.0,23643.0,141,20,12581,...,False,False,False,True,False,False,True,False,False,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,0.00,2902.0,19296.5,463,22,11751,...,False,False,False,False,False,True,False,False,False,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,0.00,2803.0,17469.0,263,14,19140,...,False,False,True,False,False,False,True,False,False,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [3]:
feature_notes = """
FEATURE MAP FOR LANE 2 (Refresh Priority Scoring)
==================================================
NUMERIC (22 raw + 5 engineered):
  search_volume          | median-imputed per content_type | YES
  competition            | median-imputed per content_type | YES
  cpc                    | median-imputed per content_type | YES
  word_count             | median-imputed per content_type | YES
  char_count             | median-imputed per content_type | YES
  content_age_days       | no missing                      | YES
  days_since_last_update | no missing                      | YES
  impressions_90d        | no missing                      | YES
  clicks_90d             | no missing                      | YES
  pageviews_90d          | no missing                      | YES
  sessions_90d           | no missing                      | YES
  users_90d              | no missing                      | YES
  engaged_sessions_90d   | no missing                      | YES
  ai_sessions_90d        | no missing                      | YES
  scroll_events_90d      | no missing                      | YES
  days_with_impressions  | no missing                      | YES
  days_with_sessions     | no missing                      | YES
  ctr                    | no missing (x100 %)             | YES
  avg_position           | zero->median (no-data flag)     | YES
  engagement_rate        | no missing (x100 %)             | YES
  scroll_rate            | no missing (x100 %)             | YES
  ai_traffic_pct         | no missing (x100 %)             | YES

ENGINEERED:
  ctr_gap                | min-max normalized              | YES
  imp_to_session_ratio   | epsilon-denominator             | YES
  staleness_weight       | stale days / age ratio          | YES
  engagement_per_session | per-session engagement depth    | YES
  ai_ratio               | AI session proportion           | YES

CATEGORICAL (one-hot):
  content_type           | 3 levels  | no missing  | YES
  main_intent            | 4 levels  | available   | YES
  competition_level      | 3 levels  | imputed     | YES
  age_tier               | 6 levels  | no missing  | YES
  freshness_tier         | 5 levels  | no missing  | YES
  impression_tier        | 6 levels  | no missing  | YES
  word_count_tier        | 4 levels  | imputed     | YES
  position_tier          | 6 levels  | no missing  | YES
"""
print(feature_notes)


FEATURE MAP FOR LANE 2 (Refresh Priority Scoring)
NUMERIC (22 raw + 5 engineered):
  search_volume          | median-imputed per content_type | YES
  competition            | median-imputed per content_type | YES
  cpc                    | median-imputed per content_type | YES
  word_count             | median-imputed per content_type | YES
  char_count             | median-imputed per content_type | YES
  content_age_days       | no missing                      | YES
  days_since_last_update | no missing                      | YES
  impressions_90d        | no missing                      | YES
  clicks_90d             | no missing                      | YES
  pageviews_90d          | no missing                      | YES
  sessions_90d           | no missing                      | YES
  users_90d              | no missing                      | YES
  engaged_sessions_90d   | no missing                      | YES
  ai_sessions_90d        | no missing                      | YES
  scro

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
label_derived = ['trend_direction', 'trend_pct', 'is_declining']
feature_names = set(X_num.columns) | set(X_cat.columns)
leaked = [c for c in label_derived if c in feature_names]
print(f"Check 1 - Label-derived columns in features: {leaked if leaked else 'NONE OK'}")

future = ['impressions_last_30d','clicks_last_30d','sessions_last_30d',
          'impressions_prev_30d','clicks_prev_30d','sessions_prev_30d']
in_f = [c for c in future if c in X_num.columns]
print(f"Check 2 - Future-window columns: {'NONE OK' if not in_f else 'WARNING!'}")

print(f"Check 3 - content_id/client_id: group keys only OK")
print(f"Check 4 - Using per-content-type median, not blind fillna(0) OK")
print(f"Check 5 - Provider/model columns excluded OK")
print("\n=== LEAKAGE CHECK PASSED ===")

Check 1 - Label-derived columns in features: NONE OK
Check 2 - Future-window columns: NONE OK
Check 3 - content_id/client_id: group keys only OK
Check 4 - Using per-content-type median, not blind fillna(0) OK
Check 5 - Provider/model columns excluded OK

=== LEAKAGE CHECK PASSED ===


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [5]:
excluded = """
EXCLUDED FIELDS - with rationale
1. trend_direction       | LABEL SOURCE - this IS the label
2. trend_pct             | DERIVED FROM LABEL - would leak outcome
3. impressions_last_30d  | OVERLAPPING WINDOW
4. clicks_last_30d       | OVERLAPPING WINDOW
5. sessions_last_30d     | OVERLAPPING WINDOW
6. impressions_prev_30d  | OVERLAPPING WINDOW
7. clicks_prev_30d       | OVERLAPPING WINDOW
8. sessions_prev_30d     | OVERLAPPING WINDOW
9. provider_used         | NOT PREDICTIVE
10. model_used           | NOT PREDICTIVE
11. content_id           | IDENTIFIER - would overfit per page
12. client_id            | IDENTIFIER - would overfit per client
13. is_declining         | THE LABEL - never a feature
"""
print(excluded)


EXCLUDED FIELDS - with rationale
1. trend_direction       | LABEL SOURCE - this IS the label
2. trend_pct             | DERIVED FROM LABEL - would leak outcome
3. impressions_last_30d  | OVERLAPPING WINDOW
4. clicks_last_30d       | OVERLAPPING WINDOW
5. sessions_last_30d     | OVERLAPPING WINDOW
6. impressions_prev_30d  | OVERLAPPING WINDOW
7. clicks_prev_30d       | OVERLAPPING WINDOW
8. sessions_prev_30d     | OVERLAPPING WINDOW
9. provider_used         | NOT PREDICTIVE
10. model_used           | NOT PREDICTIVE
11. content_id           | IDENTIFIER - would overfit per page
12. client_id            | IDENTIFIER - would overfit per client
13. is_declining         | THE LABEL - never a feature



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.